# Full CTR model benchmark on Kaggle 2xT4

This notebook synchronizes the evaluation branch, builds feature-engineered parquet files from the processed Kaggle input, trains all five CTR models sequentially, and evaluates validation/test predictions. It defaults to the full dataset (FIT_SAMPLE_SIZE = 0) and writes resumable artifacts, metrics, and plots.

The notebook does not commit or push generated artifacts. Model files are intentionally excluded from the result bundle because they are large.

In [ ]:
import base64
import importlib.metadata as metadata
import json
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/AIVIETNAM-AIO-Dewey/CTR-prediction-on-Alibaba-Display-Advertising-Dataset.git"
BRANCH = "develop"
GITHUB_TOKEN_SECRET = "GITHUB_TOKEN"
ON_KAGGLE = Path("/kaggle/input").exists()

def run_command(command, cwd=None, redact=(), env=None):
    shown = " ".join(str(item) for item in command)
    for secret in redact:
        shown = shown.replace(str(secret), "***")
    print("$", shown)
    return subprocess.run(command, cwd=cwd, check=True, env=env)

def git_environment(token):
    environment = os.environ.copy()
    if token:
        # Git reads this transient config from the child environment; it is
        # never persisted in .git/config or exposed in the command line.
        encoded = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        environment.update({
            "GIT_CONFIG_COUNT": "1",
            "GIT_CONFIG_KEY_0": "http.extraHeader",
            "GIT_CONFIG_VALUE_0": f"Authorization: Basic {encoded}",
        })
    return environment

if ON_KAGGLE:
    repo_url = REPO_URL
    token = None
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret(GITHUB_TOKEN_SECRET)
    except Exception:
        print("No GitHub token found; trying the repository without credentials.")

    ROOT = Path("/kaggle/working") / REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
    git_env = git_environment(token)
    if (ROOT / ".git").exists():
        run_command(["git", "-C", str(ROOT), "remote", "set-url", "origin", REPO_URL])
        run_command(["git", "-C", str(ROOT), "fetch", "origin", BRANCH], env=git_env)
        run_command(["git", "-C", str(ROOT), "switch", BRANCH], env=git_env)
        run_command(["git", "-C", str(ROOT), "pull", "--ff-only", "origin", BRANCH], env=git_env)
    else:
        run_command(["git", "clone", "--depth", "1", "--branch", BRANCH, repo_url, str(ROOT)], env=git_env)
        run_command(["git", "-C", str(ROOT), "remote", "set-url", "origin", REPO_URL])
    stored_remote = subprocess.check_output(["git", "-C", str(ROOT), "remote", "get-url", "origin"], text=True).strip()
    if token and token in stored_remote:
        raise RuntimeError("Credential unexpectedly persisted in git remote configuration.")
    token = None
    git_env = None
else:
    ROOT = Path.cwd()
    while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
        ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
GIT_COMMIT = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
try:
    gpu_info = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True).strip()
except (FileNotFoundError, subprocess.CalledProcessError):
    gpu_info = "nvidia-smi unavailable"
gpu_count = len([line for line in gpu_info.splitlines() if line.strip()]) if gpu_info != "nvidia-smi unavailable" else 0
if ON_KAGGLE and gpu_count < 2:
    raise RuntimeError(f"Expected two Kaggle GPUs, detected {gpu_count}: {gpu_info}")
try:
    import psutil
    memory_info = f"{psutil.virtual_memory().total / 1e9:.1f} GB RAM"
except ImportError:
    memory_info = "RAM information unavailable"
print(f"Environment: {'Kaggle' if ON_KAGGLE else 'local'}")
print(f"Repository: {ROOT}\nCommit: {GIT_COMMIT}")
print(f"GPU(s): {gpu_info}\n{memory_info}")

In [ ]:
try:
    from packaging.requirements import Requirement
    from packaging.version import Version
except ImportError:
    run_command([sys.executable, "-m", "pip", "install", "-q", "packaging>=23"])
    from packaging.requirements import Requirement
    from packaging.version import Version

REQUIRED = {
    "numpy": "numpy>=1.24", "pandas": "pandas>=2.0",
    "polars": "polars>=0.20", "pyarrow": "pyarrow>=14",
    "scipy": "scipy>=1.10", "sklearn": "scikit-learn>=1.3",
    "joblib": "joblib>=1.3", "psutil": "psutil>=5.9", "yaml": "pyyaml>=6.0",
    "lightgbm": "lightgbm>=4.0", "xgboost": "xgboost>=2.0",
    "catboost": "catboost>=1.2", "duckdb": "duckdb>=1.0", "matplotlib": "matplotlib>=3.7",
}
missing = []
for module, requirement in REQUIRED.items():
    try:
        parsed = Requirement(requirement)
        installed = Version(metadata.version(parsed.name))
        if parsed.specifier and installed not in parsed.specifier:
            missing.append(requirement)
    except metadata.PackageNotFoundError:
        missing.append(requirement)
if missing:
    print("Installing missing packages:", missing)
    run_command([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are already installed.")

In [ ]:
import gc
import json
import time
import polars as pl
from src.evaluate.signatures import partition_fingerprint, source_fingerprint
from src.evaluate.experiment_runner import validate_engineered_cache

def find_input_dir():
    roots = [Path("/kaggle/input")] if ON_KAGGLE else [ROOT / "data/processed"]
    for root in roots:
        if not root.exists():
            continue
        candidates = [root] if (root / "train.parquet").exists() else list(root.rglob("train.parquet"))
        for candidate in candidates:
            if all((candidate.parent / name).exists() for name in ("val.parquet", "test.parquet")):
                return candidate.parent
    raise FileNotFoundError("Could not find train.parquet, val.parquet and test.parquet in Kaggle Input.")

INPUT_DIR = find_input_dir()
FE_DIR = Path("/kaggle/working/ctr_processed_fe") if ON_KAGGLE else ROOT / "data/processed"
FE_DIR.mkdir(parents=True, exist_ok=True)
FE_FILES = [FE_DIR / name for name in ("train_fe.parquet", "val_fe.parquet", "test_fe.parquet")]
metadata_path = FE_DIR / "feature_metadata.json"
FEATURE_CONFIG_PATH = ROOT / "configs/feature_engineering.yaml"
FEATURE_SOURCE_FINGERPRINT = source_fingerprint([
    ROOT / "src/features/feature_engineer.py",
    ROOT / "src/features/run_feature_engineering.py",
])
INPUT_FINGERPRINT = partition_fingerprint(INPUT_DIR)
cache_valid, cached_metadata, INPUT_FINGERPRINT = validate_engineered_cache(
    INPUT_DIR, FE_DIR, FEATURE_CONFIG_PATH,
    feature_source_fingerprint=FEATURE_SOURCE_FINGERPRINT,
)
if not cache_valid:
    print(f"Building engineered partitions from {INPUT_DIR} -> {FE_DIR}")
    run_command([
        sys.executable, "-m", "src.features.run_feature_engineering",
        "--config", str(FEATURE_CONFIG_PATH),
        "--input-dir", str(INPUT_DIR), "--output-dir", str(FE_DIR),
        "--memory-bounded",
    ], cwd=ROOT)
else:
    print(f"Using existing engineered partitions in {FE_DIR}")
for path in FE_FILES:
    if not path.exists():
        raise FileNotFoundError(path)
    print(f"{path.name}: {path.stat().st_size / 1e9:.2f} GB")
with metadata_path.open(encoding="utf-8") as handle:
    FEATURE_METADATA = json.load(handle)
print("Feature metadata columns:", len(FEATURE_METADATA.get("columns", [])))

In [ ]:
import copy
import yaml
from src.evaluate.signatures import (
    build_evaluation_signature, build_training_signature, config_fingerprint,
    package_versions, partition_fingerprint, source_fingerprint,
)

FIT_SAMPLE_SIZE = 0
EVAL_SAMPLE_SIZE = 0
RANDOM_SEED = 42
THRESHOLDS = [0.05, 0.10, 0.20, 0.30, 0.50]
MODELS_TO_RUN = ["logistic_regression", "lightgbm", "xgboost", "catboost", "random_forest"]
GPU_AVAILABLE = subprocess.run(["nvidia-smi"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0
USE_GPU = ON_KAGGLE and GPU_AVAILABLE

MODELS_DIR = ROOT / "models"
EXPERIMENTS_DIR = ROOT / "experiments"
PLOTS_DIR = ROOT / "outputs" / "model_evaluation"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
EXPERIMENTS_DIR.mkdir(parents=True, exist_ok=True)

def prepare_config(model_key):
    path = ROOT / "configs" / f"{model_key}.yaml"
    with path.open(encoding="utf-8") as handle:
        config = yaml.safe_load(handle) or {}
    config["source_path"] = str(path)
    config["paths"]["processed_dir"] = str(FE_DIR)
    config["paths"]["models_dir"] = str(MODELS_DIR)
    config["paths"]["manifest_output"] = str(EXPERIMENTS_DIR / f"{model_key}_run.json")
    config["data"]["use_fe"] = True
    config["data"]["sample_size"] = FIT_SAMPLE_SIZE
    config["data"]["sample_fraction"] = None
    config["data"]["random_seed"] = RANDOM_SEED
    if model_key == "logistic_regression" and FIT_SAMPLE_SIZE == 0:
        config["params"].update({"use_sgd": True, "solver": "sgd"})
    if model_key == "xgboost" and USE_GPU:
        config["params"]["device"] = "cuda:0"
    if model_key == "catboost" and USE_GPU:
        config["params"].update({"task_type": "GPU", "devices": "0:1", "rsm": 1.0})
    return config

ALL_MODEL_KEYS = ["logistic_regression", "lightgbm", "xgboost", "catboost", "random_forest"]
CONFIGS = {model_key: prepare_config(model_key) for model_key in ALL_MODEL_KEYS}
FE_FINGERPRINT = partition_fingerprint(FE_DIR, suffix="_fe")
MODEL_PACKAGE_NAMES = {
    "logistic_regression": ["scikit-learn", "joblib"],
    "lightgbm": ["lightgbm", "joblib"],
    "xgboost": ["xgboost", "joblib"],
    "catboost": ["catboost", "joblib"],
    "random_forest": ["scikit-learn", "joblib"],
}
MODEL_PACKAGE_VERSIONS = {
    key: package_versions(names) for key, names in MODEL_PACKAGE_NAMES.items()
}
TRAIN_SPLITS = {
    "logistic_regression": ["train"],
    "random_forest": ["train"],
    "lightgbm": ["train", "val"],
    "xgboost": ["train", "val"],
    "catboost": ["train", "val"],
}
MODEL_SOURCE_FILES = {
    model_key: [
        ROOT / "src/models/train.py",
        ROOT / "src/models/data_utils.py",
        ROOT / "src/models" / ("logistic_regression_model.py" if model_key == "logistic_regression" else f"{model_key}_model.py"),
        ROOT / "configs" / f"{model_key}.yaml",
    ]
    for model_key in ALL_MODEL_KEYS
}
EVALUATION_SOURCE_FINGERPRINT = source_fingerprint([
    ROOT / "src/evaluate/metrics.py",
    ROOT / "src/evaluate/evaluator.py",
    ROOT / "src/evaluate/plot_results.py",
])
def training_signature(model_key):
    config = CONFIGS[model_key]
    return build_training_signature({
        "model": model_key,
        "config": config,
        "config_fingerprint": config_fingerprint(config["source_path"]),
        "feature_names": FEATURE_METADATA.get("columns", []),
        "feature_schema": FE_FINGERPRINT["train"]["schema"],
        "feature_config_fingerprint": config_fingerprint(FEATURE_CONFIG_PATH),
        "feature_source_fingerprint": FEATURE_SOURCE_FINGERPRINT,
        "train_fingerprint": FE_FINGERPRINT["train"],
        "validation_fingerprint": FE_FINGERPRINT["val"] if "val" in TRAIN_SPLITS[model_key] else None,
        "fit_sample_size": FIT_SAMPLE_SIZE,
        "random_seed": RANDOM_SEED,
        "gpu_overrides": {key: config["params"].get(key) for key in ("device", "task_type", "devices", "rsm") if key in config["params"]},
        "package_versions": MODEL_PACKAGE_VERSIONS[model_key],
        "source_fingerprint": MODEL_SOURCE_FILES[model_key] and source_fingerprint(MODEL_SOURCE_FILES[model_key]),
    })
TRAINING_SIGNATURES = {model_key: training_signature(model_key) for model_key in ALL_MODEL_KEYS}
def evaluation_signature(model_key):
    return build_evaluation_signature({
        "training_signature": TRAINING_SIGNATURES[model_key],
        "validation_fingerprint": FE_FINGERPRINT["val"],
        "test_fingerprint": FE_FINGERPRINT["test"],
        "eval_sample_size": EVAL_SAMPLE_SIZE,
        "random_seed": RANDOM_SEED,
        "thresholds": THRESHOLDS,
        "source_fingerprint": EVALUATION_SOURCE_FINGERPRINT,
    })
EVALUATION_SIGNATURES = {model_key: evaluation_signature(model_key) for model_key in ALL_MODEL_KEYS}
RUN_SIGNATURES = TRAINING_SIGNATURES  # compatibility alias for older notebook cells
print("Models:", list(CONFIGS))
print("Full-data training:", FIT_SAMPLE_SIZE == 0, "GPU overrides:", USE_GPU)

In [ ]:
from src.evaluate.evaluator import (
    evaluate_predictions,
    write_evaluation_outputs,
)
from src.evaluate.experiment_runner import (
    load_compatible_results, load_experiment_state, load_valid_checkpoint,
    release_iteration_state, write_experiment_state,
)
from src.models.data_utils import load_ctr_partition
from src.models.train import fit_from_config, get_model_class

TRAIN_ROWS = FE_FINGERPRINT["train"]["rows"]
RESULTS_PATH = EXPERIMENTS_DIR / "model_evaluation_results.json"
STATE_PATH = EXPERIMENTS_DIR / "experiment_state.json"
results_by_model = load_compatible_results(RESULTS_PATH, EVALUATION_SIGNATURES)
state = load_experiment_state(STATE_PATH)
if state.get("evaluation_signatures") != EVALUATION_SIGNATURES:
    state = {}
state.update({"evaluation_signatures": EVALUATION_SIGNATURES, "requested_models": list(MODELS_TO_RUN)})
state.setdefault("failed_models", {})
state["completed_models"] = sorted(results_by_model)
write_experiment_state(STATE_PATH, state)
errors = {key: value for key, value in dict(state.get("failed_models", {})).items() if key in MODELS_TO_RUN}
print("Reusable evaluation results:", sorted(results_by_model))

def artifact_path(model_key):
    config = CONFIGS[model_key]
    basename = config["paths"].get("model_basename", model_key)
    return MODELS_DIR / f"{basename}_fe.joblib"

def manifest_path(model_key):
    return EXPERIMENTS_DIR / f"{model_key}_run.json"

def expected_manifest_params(model_key):
    params = copy.deepcopy(CONFIGS[model_key].get("params", {}))
    params.pop("early_stopping_rounds", None)
    params.pop("verbose_eval", None)
    return params

def expected_sampling(model_key):
    return {"sample_size": None if FIT_SAMPLE_SIZE <= 0 else FIT_SAMPLE_SIZE, "sample_fraction": None, "random_seed": RANDOM_SEED}

def expected_gpu_overrides(model_key):
    params = CONFIGS[model_key].get("params", {})
    return {key: params[key] for key in ("device", "task_type", "devices", "rsm") if key in params}

def expected_feature_names(model_key):
    features = CONFIGS[model_key].get("features", {})
    excluded = set(features.get("exclude_cols") or []) | set(features.get("drop_features") or []) | {features.get("target", "clk")}
    return [name for name in FEATURE_METADATA["columns"] if name not in excluded]

def valid_checkpoint(model_key):
    def expected_rows(split):
        if split not in TRAIN_SPLITS[model_key]:
            return 0
        full_rows = FE_FINGERPRINT[split]["rows"]
        return min(full_rows, FIT_SAMPLE_SIZE) if FIT_SAMPLE_SIZE > 0 else full_rows
    rows = {"train": expected_rows("train"), "val": expected_rows("val"), "test": 0}
    return load_valid_checkpoint(
        model_key, artifact_path(model_key), manifest_path(model_key),
        get_model_class(model_key).load, expected_rows=rows,
        expected_training_signature=TRAINING_SIGNATURES[model_key],
        expected_feature_names=expected_feature_names(model_key),
        expected_params=expected_manifest_params(model_key),
        expected_sampling=expected_sampling(model_key),
        expected_gpu_overrides=expected_gpu_overrides(model_key),
    )

def persist_outputs():
    if not results_by_model:
        return
    write_evaluation_outputs(
        list(results_by_model.values()), EXPERIMENTS_DIR, PLOTS_DIR,
        metadata={"git_commit": GIT_COMMIT, "processed_dir": str(FE_DIR), "input_dir": str(INPUT_DIR), "fit_sample_size": FIT_SAMPLE_SIZE, "eval_sample_size": EVAL_SAMPLE_SIZE, "random_seed": RANDOM_SEED, "gpu": USE_GPU, "gpu_info": gpu_info, "requested_models": list(MODELS_TO_RUN), "completed_models": sorted(results_by_model), "failed_models": dict(errors), "training_signatures": TRAINING_SIGNATURES, "evaluation_signatures": EVALUATION_SIGNATURES},
    )

for model_key in MODELS_TO_RUN:
    started = time.time()
    run = model = X_val = X_test = p_val = p_test = y_val = y_test = None
    try:
        config, artifact, manifest = CONFIGS[model_key], artifact_path(model_key), manifest_path(model_key)
        checkpoint = valid_checkpoint(model_key)
        if model_key in results_by_model and checkpoint is not None:
            print(f"[{model_key}] reusable evaluation result found; skipping")
            errors.pop(model_key, None)
            state["completed_models"] = sorted(results_by_model)
            state["failed_models"] = dict(errors)
            write_experiment_state(STATE_PATH, state)
            persist_outputs()
            continue
        if checkpoint is not None:
            print(f"[{model_key}] valid checkpoint found; skipping fit")
            model = checkpoint
        else:
            print(f"[{model_key}] fitting with splits {TRAIN_SPLITS[model_key]}")
            run = fit_from_config(
                config=config, config_path=config["source_path"], model_key=model_key,
                processed_dir=str(FE_DIR), models_dir=str(MODELS_DIR),
                sample_size=FIT_SAMPLE_SIZE, sample_fraction=None, random_seed=RANDOM_SEED,
                training_signature=TRAINING_SIGNATURES[model_key],
                splits=TRAIN_SPLITS[model_key],
                save_artifact=True, write_manifest=True,
            )
            model = run.model
            X_val, y_val = run.dataset.X_val, run.dataset.y_val
            run.dataset.X_train = run.dataset.y_train = None
            run.dataset.X_val = run.dataset.y_val = None
            run.dataset.X_test = run.dataset.y_test = None
            run.dataset = None
            run = None
            gc.collect()
        if X_val is None:
            X_val, y_val = load_ctr_partition(str(FE_DIR), "val", model.feature_names, use_fe=True, sample_size=EVAL_SAMPLE_SIZE, random_seed=RANDOM_SEED)
        p_val = model.predict_proba(X_val)
        X_val = None
        gc.collect()
        X_test, y_test = load_ctr_partition(str(FE_DIR), "test", model.feature_names, use_fe=True, sample_size=EVAL_SAMPLE_SIZE, random_seed=RANDOM_SEED)
        p_test = model.predict_proba(X_test)
        X_test = None
        gc.collect()
        result = evaluate_predictions(
            model_key, y_val, p_val, y_test, p_test, THRESHOLDS,
            artifact_path=str(artifact), manifest_path=str(manifest),
            metadata={"git_commit": GIT_COMMIT, "training_signature": TRAINING_SIGNATURES[model_key], "evaluation_signature": EVALUATION_SIGNATURES[model_key], "fit_sample_size": FIT_SAMPLE_SIZE, "eval_sample_size": EVAL_SAMPLE_SIZE, "gpu": USE_GPU, "elapsed_seconds": round(time.time() - started, 2)},
        )
        result.metadata["training_signature"] = TRAINING_SIGNATURES[model_key]
        result.metadata["evaluation_signature"] = EVALUATION_SIGNATURES[model_key]
        results_by_model[model_key] = result
        errors.pop(model_key, None)
        state["completed_models"] = sorted(results_by_model)
        state["failed_models"] = dict(errors)
        write_experiment_state(STATE_PATH, state)
        persist_outputs()
        print(f"[{model_key}] complete in {time.time() - started:.1f}s; val ROC-AUC={result.validation.probability_metrics['roc_auc']:.5f}")
    except Exception as exc:
        errors[model_key] = repr(exc)
        state["completed_models"] = sorted(results_by_model)
        state["failed_models"] = dict(errors)
        write_experiment_state(STATE_PATH, state)
        persist_outputs()
        print(f"[{model_key}] FAILED: {exc}")
    finally:
        release_iteration_state(globals())

results = list(results_by_model.values())
if results:
    persist_outputs()
if errors:
    raise RuntimeError(f"Model runs failed: {errors}")
print(f"Completed {len(results)} model evaluations.")

In [ ]:
import zipfile

ranking = sorted(results, key=lambda item: (-item.validation.probability_metrics["roc_auc"], item.validation.probability_metrics["log_loss"]))
print("Validation ranking:")
for position, result in enumerate(ranking, start=1):
    metrics = result.validation.probability_metrics
    print(f"{position}. {result.model_key:20s} ROC-AUC={metrics['roc_auc']:.5f} LogLoss={metrics['log_loss']:.5f} PR-AUC={metrics['pr_auc']:.5f}")

bundle = ROOT / "evaluation_results_bundle.zip"
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in [EXPERIMENTS_DIR / "model_evaluation_results.json", EXPERIMENTS_DIR / "model_metrics.csv", EXPERIMENTS_DIR / "threshold_metrics.csv", EXPERIMENTS_DIR / "experiment_state.json"]:
        if path.exists():
            archive.write(path, path.relative_to(ROOT))
    for path in PLOTS_DIR.glob("*.png"):
        archive.write(path, path.relative_to(ROOT))
    for model_key in sorted(results_by_model):
        path = manifest_path(model_key)
        if path.exists():
            archive.write(path, path.relative_to(ROOT))
print(f"Result bundle: {bundle} ({bundle.stat().st_size / 1e6:.2f} MB)")